In [1]:
# ============================================================
# TRUSTSYN TRUST LAYER v1
# Ensemble Agreement + Uncertainty Score
# ============================================================

import pandas as pd
import numpy as np
from pathlib import Path


# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

BASE = Path("/Users/konuri/stacking")

STACK_DIR = BASE / "STACKING_TABLES"

OUT_DIR = BASE / "TrustSyn_TRUST_LAYER"
OUT_DIR.mkdir(exist_ok=True)


# ------------------------------------------------------------
# LOAD FROZEN STACKING TABLES
# ------------------------------------------------------------

files = {
    "RANDOM": STACK_DIR / "RANDOM_STACKING_TABLE.csv",
    "COLD_COMBINATION": STACK_DIR / "COLD_COMBINATION_STACKING_TABLE.csv",
    "COLD_CELL": STACK_DIR / "COLD_CELL_STACKING_TABLE.csv"
}


all_results = []


for split, path in files.items():

    print("\n====================")
    print(split)

    df = pd.read_csv(path)

    print("Loaded:", df.shape)


    # --------------------------------------------------------
    # Required frozen predictions
    # --------------------------------------------------------

    required = [
        "catboost_prediction",
        "dmpnn_prediction",
        "y_true"
    ]

    missing = [
        c for c in required
        if c not in df.columns
    ]

    if missing:
        raise Exception(
            f"{split} missing columns: {missing}"
        )


    # --------------------------------------------------------
    # Build ensemble
    #
    # Currently:
    #   CatBoost
    #   D-MPNN
    #
    # Later we can add CatBoost seeds
    # --------------------------------------------------------

    ensemble_cols = [
        "catboost_prediction",
        "dmpnn_prediction"
    ]


    df["ensemble_mean"] = (
        df[ensemble_cols]
        .mean(axis=1)
    )


    df["ensemble_std"] = (
        df[ensemble_cols]
        .std(axis=1)
        .fillna(0)
    )


    # Agreement score
    # Higher = models agree more

    df["agreement_score"] = (
        1 /
        (1 + df["ensemble_std"])
    )


    # uncertainty flag

    df["uncertainty_flag"] = np.where(
        df["ensemble_std"] >
        df["ensemble_std"].median(),
        "HIGH_UNCERTAINTY",
        "LOW_UNCERTAINTY"
    )


    df["split"] = split


    all_results.append(df)



# ------------------------------------------------------------
# COMBINE
# ------------------------------------------------------------

trust_df = pd.concat(
    all_results,
    ignore_index=True
)


print("\nFINAL TRUST TABLE")
print(trust_df.shape)


print(
    trust_df[
        [
            "catboost_prediction",
            "dmpnn_prediction",
            "ensemble_mean",
            "ensemble_std",
            "agreement_score",
            "uncertainty_flag"
        ]
    ].head()
)


# ------------------------------------------------------------
# SAVE
# ------------------------------------------------------------

output = (
    OUT_DIR /
    "TrustSyn_ensemble_uncertainty.csv"
)


trust_df.to_csv(
    output,
    index=False
)


print("\nSAVED:")
print(output)


RANDOM
Loaded: (29408, 74)

COLD_COMBINATION
Loaded: (29558, 74)

COLD_CELL
Loaded: (29787, 74)

FINAL TRUST TABLE
(88753, 79)
   catboost_prediction  dmpnn_prediction  ensemble_mean  ensemble_std  \
0            -1.731061         -0.069993      -0.900527      1.174552   
1            -0.114627          0.204334       0.044853      0.225539   
2            -0.183730         -2.355772      -1.269751      1.535866   
3            -0.776740         -3.475799      -2.126269      1.908523   
4            -0.781407         -0.812224      -0.796816      0.021791   

   agreement_score  uncertainty_flag  
0         0.459865   LOW_UNCERTAINTY  
1         0.815967   LOW_UNCERTAINTY  
2         0.394343  HIGH_UNCERTAINTY  
3         0.343817  HIGH_UNCERTAINTY  
4         0.978674   LOW_UNCERTAINTY  

SAVED:
/Users/konuri/stacking/TrustSyn_TRUST_LAYER/TrustSyn_ensemble_uncertainty.csv
